In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import copy
import numpy as np
import pandas as pd

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

from sklearn.model_selection import train_test_split
from torchvision import datasets
from torchvision import transforms

import time

start_time = time.time()

In [ ]:
#!pip install torch torchvision torchaudio

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# 1. Get pretrained weights for ViT-Base
pretrained_vit_weights = torchvision.models.ViT_B_16_Weights.DEFAULT # requires torchvision >= 0.13, "DEFAULT" means best available

# 2. Setup a ViT model instance with pretrained weights
pretrained_vit = torchvision.models.vit_b_16(weights=pretrained_vit_weights).to(device)

# 3. Freeze the base parameters
for parameter in pretrained_vit.parameters():
    parameter.requires_grad = False


pretrained_vit_transforms = pretrained_vit_weights.transforms()
print(pretrained_vit_transforms)

In [ ]:
num_classes = 4  # Replace with the number of classes in your dataset
batch_size = 32
learning_rate = 0.001

In [ ]:
import torch
from torchvision import transforms, datasets

# Modify the dataset path to the correct location
dataset_path = "../../Smartbin/Crawl_dataset_4_classes/test"

# Define data transformations
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Load the dataset using ImageFolder
dataset = datasets.ImageFolder(root=dataset_path, transform=data_transforms)

# Check if dataset is loaded successfully
if len(dataset) > 0:
    print(f"Dataset loaded successfully! Found {len(dataset)} images.")
else:
    print("Error: Dataset is empty. Please check the directory path and image formats.")

# Create data loader for training or testing
data_loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

In [ ]:
dataset = "../../Smartbin/data/trash_dataset"
datasets_data = datasets.ImageFolder(
    dataset,
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
)

In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import random
from torch.utils.data import Subset
random_seed = 33
random.seed(random_seed)
torch.manual_seed(random_seed)

dataset1 = {}

# train-validation split
train_idx, valtest_idx = train_test_split(list(range(len(datasets_data))),
                                          test_size=0.3,
                                          random_state=random_seed)

dataset1['train'] = Subset(datasets_data, train_idx)
valtest          = Subset(datasets_data, valtest_idx)


# validation-test split
val_idx, test_idx = train_test_split(list(range(len(valtest))),
                                     test_size=0.67,
                                     random_state=random_seed)

dataset1['valid'] = Subset(valtest, val_idx)
dataset1['test']  = Subset(valtest, test_idx)

max_trn_corrects = len(dataset1['train'])
max_val_corrects = len(dataset1['valid'])

In [ ]:
dataloaders, batch_num = {}, {}

dataloaders['train'] = DataLoader(dataset1['train'],
                                  batch_size=batch_size, shuffle=True,
                                  num_workers=12)
dataloaders['valid'] = DataLoader(dataset1['valid'],
                                  batch_size=batch_size, shuffle=True,
                                  num_workers=12)
dataloaders['test'] = DataLoader(dataset1['test'],
                                  batch_size=batch_size, shuffle=True,
                                  num_workers=12)

batch_num['train'], batch_num['valid'], batch_num['test'] = len(dataloaders['train']), len(dataloaders['valid']), len(dataloaders['test'])


print('batch_size : {}\ntrain/valid/test : {}/{}/{}'
      .format(batch_size,
              batch_num['train'], batch_num['valid'], batch_num['test']))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
imgtest = None
for images, labels in dataloaders['train']:
    imgtest = images[3]
    print(imgtest.shape)
    break

imgtest = imgtest.numpy()
imgtest = np.moveaxis(imgtest, 0, -1)
plt.imshow(imgtest)

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': []}

In [ ]:
def train_model(model, criterion, optimizer, dataloaders, num_epochs, patience):
    best_loss = float('inf')
    epochs_no_improve = 0
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch + 1, num_epochs))
        print('-' * 10)

        for phase in ['train', 'valid']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)

            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_accuracy'].append(epoch_acc)
            else:
                history['val_accuracy'].append(epoch_acc)
                history['val_loss'].append(epoch_loss)

            print('{} Loss: {:.4f} Acc: {:.4f}'.format(phase, epoch_loss, epoch_acc))

            if phase == 'valid':
                if epoch_loss < best_loss:
                    best_loss = epoch_loss
                    best_model_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1

        if epochs_no_improve == patience:
            print("Early stopping triggered. No improvement in validation loss for {} epochs.".format(patience))
            break

    model.load_state_dict(best_model_wts)
    return model


In [ ]:
model = pretrained_vit

model.heads = nn.Linear(in_features=768, out_features=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
num_epochs = 200
patience = 50

start_time = time.time()
model_trained = train_model(model, criterion, optimizer, dataloaders, num_epochs,patience)
end_time = time.time()
execution_time = end_time - start_time
print("Thời gian huấn luyện model là:", execution_time, "s")

In [ ]:
def plot_training_history(history):
    train_loss = history['train_loss']
    val_loss = history['val_loss']
    train_accuracy = history['train_accuracy']
    val_accuracy = history['val_accuracy']
    epochs = range(len(train_loss))

    # Move tensors from CUDA device to CPU if they are tensors
    if isinstance(train_accuracy[0], torch.Tensor):
        train_accuracy = [acc.cpu() for acc in train_accuracy]
    if isinstance(val_accuracy[0], torch.Tensor):
        val_accuracy = [acc.cpu() for acc in val_accuracy]

    # Plot loss
    plt.plot(epochs, train_loss, label='Training Loss')
    plt.plot(epochs, val_loss, label='Validation Loss')
    plt.title('Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

    # Plot accuracy
    plt.figure()
    plt.plot(epochs, train_accuracy, label='Training Accuracy')
    plt.plot(epochs, val_accuracy, label='Validation Accuracy')
    plt.title('Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.show()

plot_training_history(history)

In [ ]:
# Lấy danh sách các lớp từ tập dữ liệu train (hoặc bất kỳ tập dữ liệu nào khác trong dataset1)
all_classes = dataset1['train'].dataset.classes

# In ra danh sách các lớp
print("Danh sách các lớp:", all_classes)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Chuyển mô hình sang chế độ đánh giá (evaluation mode)
model.eval()

# Lấy tên các lớp từ tập dữ liệu
class_names = all_classes

# Lấy 10 ảnh và nhãn tương ứng từ tập dữ liệu test
num_images_to_display = 10
images_to_display = []
labels_to_display = []

for images, labels in dataloaders['test']:
    for i in range(len(labels)):
        images_to_display.append(images[i])
        labels_to_display.append(labels[i].item())
    if len(images_to_display) >= num_images_to_display:
        break

# Dự đoán và hiển thị 10 ảnh cùng với dự đoán và tên lớp
with torch.no_grad():
    plt.figure(figsize=(15, 8))
    for i in range(num_images_to_display):
        image = images_to_display[i].unsqueeze(0).to(device)  # Thêm chiều batch và chuyển đến device
        label = labels_to_display[i]

        output = model(image)
        _, predicted = torch.max(output, 1)
        predicted_label = predicted.item()

        # Hiển thị ảnh và dự đoán
        plt.subplot(2, 5, i + 1)
        plt.imshow(np.transpose(images_to_display[i], (1, 2, 0)))
        plt.title(f'Predicted: {class_names[predicted_label]},\n Actual: {class_names[label]}')
        plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in dataloaders['test']:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_predictions.extend(predicted.tolist())
        all_labels.extend(labels.tolist())

conf_matrix = confusion_matrix(all_labels, all_predictions)

plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, cmap='Blues', fmt='d', xticklabels=class_names, yticklabels=class_names)

for i in range(len(conf_matrix)):
    for j in range(len(conf_matrix)):
        plt.text(j + 0.5, i + 0.5, conf_matrix[i][j], ha='center', va='center', color='black')

plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
report = classification_report(all_labels, all_predictions, target_names=class_names, output_dict=True)
df = pd.DataFrame(report).transpose()
df

In [ ]:
print("Thời gian huấn luyện model là:", execution_time, "s")

In [ ]:
import timm
from huggingface_hub import hf_hub_download
import torch


saved_path = hf_hub_download('timm/coatnet_rmlp_2_rw_224.sw_in12k_ft_in1k', 'pytorch_model.bin', library_name='timm', library_version=timm.__version__)
state_dict = torch.load(saved_path)

In [ ]:
import timm
import torch

model = timm.create_model('coatnet_rmlp_2_rw_224', pretrained=False)
model.load_state_dict(state_dict)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [ ]:
model.head.fc = torch.nn.Linear(model.head.fc.in_features, num_classes)  # Thay đổi lớp đầu ra
model.head.fc = model.head.to(device)  # Chuyển lớp mới sang GPU

In [ ]:
for param in model.parameters():
    param.requires_grad = False

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
num_epochs = 200
patience = 50

start_time = time.time()
model_trained = train_model(model, criterion, optimizer, dataloaders, num_epochs,patience)
end_time = time.time()
execution_time = end_time - start_time
print("Thời gian huấn luyện model là:", execution_time, "s")